In [ ]:
import torch
from torch import nn, einsum
import torch.nn.functional as F
from torch import optim
import numpy as np
import torchvision
from torch.utils.data import Dataset, DataLoader, Subset
from einops import rearrange, repeat
from einops.layers.torch import Rearrange
from scipy.io import savemat
from scipy.spatial.distance import pdist, squareform
import os
from PIL import Image
from torchvision import transforms

class ModifiedMNIST_train(Dataset):
    def __init__(self, original_dataset, save_folder):
        self.original_dataset = original_dataset
        self.save_folder = save_folder
        self.processed_files = {}
        os.makedirs(save_folder, exist_ok=True)
        for idx in range(len(original_dataset)):
            img, label = original_dataset[idx]
            img_np = img.numpy().squeeze()
            img_large = np.array(Image.fromarray(img_np).resize((81, 81), Image.NEAREST))
            img_binary = (img_large >= 0.2).astype(np.float32)
            img_large = img_binary
            img_uint8 = (img_large * 255).astype(np.uint8)
            img_small = np.array(Image.fromarray(img_uint8).resize((28, 28), Image.NEAREST))
            filename = f"train_{idx}_{label}.png"
            filepath = os.path.join(save_folder, filename)
            Image.fromarray(img_small).save(filepath)
            self.processed_files[idx] = (filepath, label)

    def __len__(self):
        return len(self.original_dataset)

    def __getitem__(self, idx):
        filepath, label = self.processed_files[idx]
        img = Image.open(filepath).convert('L')
        img_np = np.array(img) / 255.0
        img_tensor = torch.from_numpy(img_np).unsqueeze(0).float()
        return img_tensor, label

class ModifiedMNIST_test(Dataset):
    def __init__(self, original_dataset, save_folder, noise_level):
        self.original_dataset = original_dataset
        self.save_folder = save_folder
        self.noise_level = noise_level
        self.processed_files = {}
        os.makedirs(save_folder, exist_ok=True)
        for idx in range(len(original_dataset)):
            img, label = original_dataset[idx]
            img_np = img.numpy().squeeze()
            img_large = np.array(Image.fromarray(img_np).resize((81, 81), Image.NEAREST))
            img_binary = (img_large >= 0.2).astype(np.float32)
            h, w = img_binary.shape
            num_pixels = h * w
            if self.noise_level > 0:
                flip_indices = np.random.choice(num_pixels,
                                                size=int(num_pixels * self.noise_level),
                                                replace=False)
                img_binary_flat = img_binary.ravel()
                img_binary_flat[flip_indices] = 1 - img_binary_flat[flip_indices]
                img_binary = img_binary_flat.reshape(h, w)
            img_uint8 = (img_binary * 255).astype(np.uint8)
            img_small = np.array(Image.fromarray(img_uint8).resize((28, 28), Image.NEAREST))
            filename = f"test_{idx}_{label}_size{noise_level}.png"
            filepath = os.path.join(save_folder, filename)
            Image.fromarray(img_small).save(filepath)
            self.processed_files[idx] = (filepath, label)

    def __len__(self):
        return len(self.original_dataset)

    def __getitem__(self, idx):
        filepath, label = self.processed_files[idx]
        img = Image.open(filepath).convert('L')
        img_np = np.array(img) / 255.0
        img_tensor = torch.from_numpy(img_np).unsqueeze(0).float()
        return img_tensor, label

class TransformedDataset(Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        if self.transform:
            img_pil = transforms.ToPILImage()(img.squeeze(0))
            img = self.transform(img_pil)
        return img, label

def pair(t):
    return t if isinstance(t, tuple) else (t, t)

class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fn = fn
    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)

class Attention(nn.Module):
    def __init__(self, dim, heads=4, dim_head=64, dropout=0.):
        super().__init__()
        inner_dim = dim_head * heads
        project_out = not (heads == 1 and dim_head == dim)
        self.heads = heads
        self.scale = dim_head ** -0.5
        self.attend = nn.Softmax(dim=-1)
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, dim),
            nn.Dropout(dropout)
        ) if project_out else nn.Identity()

    def forward(self, x):
        b, n, _, h = *x.shape, self.heads
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=h), qkv)
        dots = einsum('b h i d, b h j d -> b h i j', q, k) * self.scale
        attn = self.attend(dots)
        out = einsum('b h i j, b h j d -> b h i d', attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)

class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout=0.):
        super().__init__()
        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                PreNorm(dim, Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout)),
                PreNorm(dim, FeedForward(dim, mlp_dim, dropout=dropout))
            ]))
    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x
            x = ff(x) + x
        return x

class ViT(nn.Module):
    def __init__(self, *, image_size, patch_size, num_classes, dim, depth, heads, mlp_dim, pool='cls', channels=3, dim_head=64, dropout=0., emb_dropout=0.):
        super().__init__()
        image_height, image_width = pair(image_size)
        patch_height, patch_width = pair(patch_size)
        assert image_height % patch_height == 0 and image_width % patch_width == 0
        num_patches = (image_height // patch_height) * (image_width // patch_width)
        patch_dim = channels * patch_height * patch_width
        assert pool in {'cls', 'mean'}
        self.to_patch_embedding = nn.Sequential(
            Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=patch_height, p2=patch_width),
            nn.Linear(patch_dim, dim),
        )
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, dim))
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.dropout = nn.Dropout(emb_dropout)
        self.transformer = Transformer(dim, depth, heads, dim_head, mlp_dim, dropout)
        self.pool = pool
        self.to_latent = nn.Identity()
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, num_classes)
        )

    def forward(self, img):
        x = self.to_patch_embedding(img)
        b, n, _ = x.shape
        cls_tokens = repeat(self.cls_token, '() n d -> b n d', b=b)
        x = torch.cat((cls_tokens, x), dim=1)
        x += self.pos_embedding[:, :(n + 1)]
        x = self.dropout(x)
        x = self.transformer(x)
        x = x.mean(dim=1) if self.pool == 'mean' else x[:, 0]
        x = self.to_latent(x)
        return self.mlp_head(x)

def train_epoch(model, optimizer, data_loader, loss_history):
    total_samples = len(data_loader.dataset)
    model.train()
    for i, (data, target) in enumerate(data_loader):
        optimizer.zero_grad()
        output = F.log_softmax(model(data), dim=1)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if i % 100 == 0:
            loss_history.append(loss.item())

def evaluate(model, data_loader, loss_history, collect_embeddings=False):
    model.eval()
    total_samples = len(data_loader.dataset)
    correct_samples = 0
    total_loss = 0
    embeddings = []
    labels = []
    handle = None

    def hook_fn(module, input, output):
        embeddings.append(output.detach().cpu())

    if collect_embeddings:
        layer_norm = model.mlp_head[0]
        handle = layer_norm.register_forward_hook(hook_fn)

    try:
        with torch.no_grad():
            for data, target in data_loader:
                output = model(data)
                loss = F.nll_loss(F.log_softmax(output, dim=1), target, reduction='sum')
                total_loss += loss.item()
                pred = output.argmax(dim=1)
                correct_samples += pred.eq(target).sum().item()
                if collect_embeddings:
                    labels.append(target.cpu())
    finally:
        if handle is not None:
            handle.remove()

    if collect_embeddings:
        embeddings = torch.cat(embeddings, dim=0) if embeddings else torch.Tensor()
        labels = torch.cat(labels, dim=0) if labels else torch.Tensor()

    avg_loss = total_loss / total_samples
    loss_history.append(avg_loss)
    accuracy = correct_samples / total_samples

    if collect_embeddings:
        return embeddings, labels, accuracy
    else:
        return accuracy

def create_torch_splits(num_splits=10):
    transform_base = transforms.Compose([transforms.ToTensor()])
    splits = []
    for split_num in range(num_splits):
        np.random.seed(42 + split_num)
        train_full = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform_base)
        test_full = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform_base)
        train_indices, test_indices = [], []
        for label in range(10):
            train_label_idx = np.where(np.array(train_full.targets) == label)[0]
            np.random.shuffle(train_label_idx)
            train_indices.extend(train_label_idx[:750])
            test_label_idx = np.where(np.array(test_full.targets) == label)[0]
            np.random.shuffle(test_label_idx)
            test_indices.extend(test_label_idx[:250])
        train_subset = Subset(train_full, train_indices)
        test_subset = Subset(test_full, test_indices)
        splits.append((train_subset, test_subset))
    return splits

if __name__ == "__main__":
    noise_level = 0.15
    N_EPOCHS = 50
    num_runs = 10
    BATCH_SIZE_TRAIN = 128
    BATCH_SIZE_TEST = 128

    razm_class      = np.zeros((10, N_EPOCHS+1, num_runs))
    razm_class_n    = np.zeros((10, N_EPOCHS+1, num_runs))
    rasst_m_class   = np.zeros((10, 10, N_EPOCHS+1, num_runs))
    rasst_m_class_n = np.zeros((10, 10, N_EPOCHS+1, num_runs))
    accur           = np.zeros((N_EPOCHS+1, num_runs))
    centroids_coords = np.zeros((10, 128, N_EPOCHS+1, num_runs))

    transform_mnist = transforms.Compose([
        transforms.ToTensor(),
    ])

    splits = create_torch_splits(num_splits=10)

    for run_idx in range(num_runs):
        train_subset, test_subset = splits[run_idx]
        train_folder = f'D:/Digit_ViT4/train_split30v2_{run_idx + 1}'
        test_base_folder = f'D:/Digit_ViT4/test_split30v2_{run_idx + 1}'

        train_set = ModifiedMNIST_train(train_subset, train_folder)
        train_set_transformed = TransformedDataset(train_set, transform_mnist)
        train_loader = DataLoader(train_set_transformed, batch_size=BATCH_SIZE_TRAIN, shuffle=True)

        test_folder = f'{test_base_folder}/size_{noise_level}'
        test_set = ModifiedMNIST_test(test_subset, test_folder, noise_level)
        test_set_transformed = TransformedDataset(test_set, transform_mnist)
        test_loader = DataLoader(test_set_transformed, batch_size=BATCH_SIZE_TEST, shuffle=False)

        model = ViT(image_size=28, patch_size=4, num_classes=10, channels=1, dim=128, depth=8, heads=8, mlp_dim=256)
        optimizer = optim.Adam(model.parameters(), lr=0.0005)
        train_loss_history, test_loss_history = [], []

        emb, lbl, accuracy = evaluate(model, test_loader, test_loss_history, collect_embeddings=True)

        embeddings_np = emb.numpy()
        labels_np = lbl.numpy()

        classes = np.unique(labels_np)
        class_centroids = np.array([embeddings_np[labels_np == cls].mean(axis=0) for cls in classes])
        class_sizes = np.array([np.mean(np.linalg.norm(embeddings_np[labels_np == cls] - centroid, axis=1))
                                for cls, centroid in zip(classes, class_centroids)])
        euclidean_dist = squareform(pdist(class_centroids, 'euclidean'))
        mean_all = embeddings_np.mean(axis=0)
        embeddings_centered = embeddings_np - mean_all
        cov_matrix = np.cov(embeddings_centered, rowvar=False)
        eigenvalues = np.linalg.eigvalsh(cov_matrix)
        eigenvalues = np.sort(eigenvalues)[::-1]
        total_variance = eigenvalues.sum()
        scale_factor = np.sqrt(total_variance)
        euclidean_dist_norm = euclidean_dist / scale_factor
        class_sizes_norm = class_sizes / scale_factor

        razm_class[:, 0, run_idx] = class_sizes
        razm_class_n[:, 0, run_idx] = class_sizes_norm
        rasst_m_class[:, :, 0, run_idx] = euclidean_dist
        rasst_m_class_n[:, :, 0, run_idx] = euclidean_dist_norm
        accur[0, run_idx] = accuracy
        centroids_coords[:, :, 0, run_idx] = class_centroids

        for epoch in range(1, N_EPOCHS + 1):
            train_epoch(model, optimizer, train_loader, train_loss_history)
            emb, lbl, accuracy = evaluate(model, test_loader, test_loss_history, collect_embeddings=True)

            embeddings_np = emb.numpy()
            labels_np = lbl.numpy()

            classes = np.unique(labels_np)
            class_centroids = np.array([embeddings_np[labels_np == cls].mean(axis=0) for cls in classes])
            class_sizes = np.array([np.mean(np.linalg.norm(embeddings_np[labels_np == cls] - centroid, axis=1))
                                    for cls, centroid in zip(classes, class_centroids)])
            euclidean_dist = squareform(pdist(class_centroids, 'euclidean'))
            mean_all = embeddings_np.mean(axis=0)
            embeddings_centered = embeddings_np - mean_all
            cov_matrix = np.cov(embeddings_centered, rowvar=False)
            eigenvalues = np.linalg.eigvalsh(cov_matrix)
            eigenvalues = np.sort(eigenvalues)[::-1]
            total_variance = eigenvalues.sum()
            scale_factor = np.sqrt(total_variance)
            euclidean_dist_norm = euclidean_dist / scale_factor
            class_sizes_norm = class_sizes / scale_factor

            razm_class[:, epoch, run_idx] = class_sizes
            razm_class_n[:, epoch, run_idx] = class_sizes_norm
            rasst_m_class[:, :, epoch, run_idx] = euclidean_dist
            rasst_m_class_n[:, :, epoch, run_idx] = euclidean_dist_norm
            accur[epoch, run_idx] = accuracy
            centroids_coords[:, :, epoch, run_idx] = class_centroids

            savemat('DATA_ViT.mat', {
                'razm_class': razm_class,
                'razm_class_n': razm_class_n,
                'rasst_m_class': rasst_m_class,
                'rasst_m_class_n': rasst_m_class_n,
                'accur': accur,
                'centroids_coords': centroids_coords,
            })